In [2]:
# Anime Cluster Characterization Analysis
# =====================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Load and merge data
def load_and_merge_data():
    """Load cluster and anime data, then merge them"""
    print("Loading data...")
    
    # Load cluster representatives
    clusters_df = pd.read_csv('../data/final_clusters/spectral_13_reps.csv')
    print(f"Loaded {len(clusters_df)} cluster entries")
    
    # Load anime metadata
    anime_df = pd.read_csv('../data/anime.csv')
    print(f"Loaded {len(anime_df)} anime entries")
    
    # Merge on anime_id
    merged_df = clusters_df.merge(anime_df, on='anime_id', how='left')
    print(f"Merged dataset: {len(merged_df)} entries")
    
    return merged_df

# 1. CLUSTER SIZE DISTRIBUTION
def plot_cluster_sizes(df):
    """Plot the distribution of anime across clusters"""
    plt.figure(figsize=(12, 6))
    
    cluster_counts = df['cluster_id'].value_counts().sort_index()
    
    bars = plt.bar(cluster_counts.index, cluster_counts.values, 
                   color=sns.color_palette("husl", len(cluster_counts)),
                   alpha=0.8, edgecolor='black', linewidth=0.5)
    
    plt.title('Anime Distribution Across Clusters\n(Representative Sample)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Cluster ID', fontsize=12)
    plt.ylabel('Number of Anime', fontsize=12)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                f'{int(height)}', ha='center', va='bottom', fontweight='bold')
    
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

# 2. RATING DISTRIBUTION BY CLUSTER
def plot_rating_distributions(df):
    """Plot rating distributions and averages by cluster"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Box plot of ratings by cluster
    df_with_scores = df.dropna(subset=['score'])
    
    sns.boxplot(data=df_with_scores, x='cluster_id', y='score', ax=ax1)
    ax1.set_title('Rating Distributions by Cluster', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Cluster ID')
    ax1.set_ylabel('Rating (1-10)')
    ax1.grid(axis='y', alpha=0.3)
    
    # Average ratings by cluster
    avg_ratings = df_with_scores.groupby('cluster_id')['score'].agg(['mean', 'std', 'count'])
    
    bars = ax2.bar(avg_ratings.index, avg_ratings['mean'], 
                   yerr=avg_ratings['std'], capsize=5,
                   color=sns.color_palette("husl", len(avg_ratings)),
                   alpha=0.8, edgecolor='black', linewidth=0.5)
    
    ax2.set_title('Average Ratings by Cluster', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Cluster ID')
    ax2.set_ylabel('Average Rating')
    ax2.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, (bar, mean_val) in enumerate(zip(bars, avg_ratings['mean'])):
        ax2.text(bar.get_x() + bar.get_width()/2., mean_val + 0.05,
                f'{mean_val:.2f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return avg_ratings

# 3. GENRE ANALYSIS
def analyze_genres(df):
    """Analyze genre distributions across clusters"""
    # Parse genres for each anime
    genre_data = []
    
    for _, row in df.iterrows():
        if pd.notna(row['genres']) and row['genres']:
            genres = [g.strip() for g in str(row['genres']).split('|') if g.strip()]
            for genre in genres:
                genre_data.append({
                    'cluster_id': row['cluster_id'],
                    'genre': genre,
                    'anime_id': row['anime_id']
                })
    
    genre_df = pd.DataFrame(genre_data)
    return genre_df

def plot_genre_heatmap(df):
    """Create a heatmap of genre distributions across clusters"""
    genre_df = analyze_genres(df)
    
    # Get top genres by frequency
    top_genres = genre_df['genre'].value_counts().head(15).index.tolist()
    
    # Create pivot table for heatmap
    genre_cluster_counts = genre_df[genre_df['genre'].isin(top_genres)].groupby(['cluster_id', 'genre']).size().unstack(fill_value=0)
    
    # Convert to percentages within each cluster
    cluster_sizes = df['cluster_id'].value_counts().sort_index()
    genre_percentages = genre_cluster_counts.div(cluster_sizes, axis=0) * 100
    
    plt.figure(figsize=(16, 10))
    
    # Create heatmap
    sns.heatmap(genre_percentages, 
                annot=True, 
                fmt='.1f', 
                cmap='Blues', 
                cbar_kws={'label': 'Percentage within Cluster'},
                linewidths=0.5)
    
    plt.title('Genre Distribution Across Clusters (%)\nTop 15 Most Common Genres', 
              fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Genres', fontsize=12)
    plt.ylabel('Cluster ID', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    return genre_percentages

# 4. CONTENT TYPE ANALYSIS
def plot_content_types(df):
    """Analyze content types (TV, Movie, OVA, etc.) across clusters"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Overall type distribution
    type_counts = df['type'].value_counts()
    
    wedges, texts, autotexts = ax1.pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
                                       colors=sns.color_palette("husl", len(type_counts)))
    ax1.set_title('Overall Content Type Distribution', fontsize=14, fontweight='bold')
    
    # Type distribution by cluster
    cluster_type_df = df.groupby(['cluster_id', 'type']).size().unstack(fill_value=0)
    
    cluster_type_df.plot(kind='bar', stacked=True, ax=ax2, 
                        color=sns.color_palette("husl", len(cluster_type_df.columns)))
    ax2.set_title('Content Type Distribution by Cluster', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Cluster ID')
    ax2.set_ylabel('Number of Anime')
    ax2.legend(title='Content Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.show()
    
    return cluster_type_df

# 5. CLUSTER CHARACTERIZATION SUMMARY
def create_cluster_summary(df):
    """Create a comprehensive summary of each cluster"""
    genre_df = analyze_genres(df)
    
    summary_data = []
    
    for cluster_id in sorted(df['cluster_id'].unique()):
        cluster_data = df[df['cluster_id'] == cluster_id]
        cluster_genres = genre_df[genre_df['cluster_id'] == cluster_id]
        
        # Top genres
        top_genres = cluster_genres['genre'].value_counts().head(3).to_dict()
        
        # Rating stats
        scores = cluster_data['score'].dropna()
        
        # Type distribution
        types = cluster_data['type'].value_counts().to_dict()
        
        summary_data.append({
            'Cluster': cluster_id,
            'Size': len(cluster_data),
            'Avg_Rating': scores.mean() if len(scores) > 0 else None,
            'Rating_Std': scores.std() if len(scores) > 0 else None,
            'Top_Genres': ', '.join([f"{genre} ({count})" for genre, count in top_genres.items()]),
            'Content_Types': ', '.join([f"{type_} ({count})" for type_, count in types.items()]),
            'Sample_Titles': ', '.join(cluster_data['title'].head(3).tolist())
        })
    
    summary_df = pd.DataFrame(summary_data)
    return summary_df

# 6. MAIN EXECUTION
def main():
    """Run the complete cluster characterization analysis"""
    print("🎌 ANIME CLUSTER CHARACTERIZATION ANALYSIS")
    print("=" * 50)
    
    # Load data
    df = load_and_merge_data()
    
    print(f"\n📊 Dataset Overview:")
    print(f"- Total clusters: {df['cluster_id'].nunique()}")
    print(f"- Total anime: {len(df)}")
    print(f"- Unique genres: {len(set([g.strip() for genres in df['genres'].dropna() for g in str(genres).split('|') if g.strip()]))}")
    print(f"- Content types: {df['type'].nunique()}")
    
    # Generate visualizations
    print("\n📈 Generating visualizations...")
    
    print("\n1. Cluster Size Distribution")
    plot_cluster_sizes(df)
    
    print("\n2. Rating Analysis")
    rating_stats = plot_rating_distributions(df)
    
    print("\n3. Genre Distribution Heatmap")
    genre_heatmap = plot_genre_heatmap(df)
    
    print("\n4. Content Type Analysis")
    type_distribution = plot_content_types(df)
    
    print("\n5. Cluster Summary")
    cluster_summary = create_cluster_summary(df)
    
    # Display summary table
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 50)
    
    print("\n📋 CLUSTER CHARACTERIZATION SUMMARY")
    print("=" * 80)
    print(cluster_summary.to_string(index=False))
    
    print("\n✅ Analysis complete! All visualizations generated.")
    
    return df, cluster_summary, rating_stats, genre_heatmap, type_distribution

# Run the analysis
if __name__ == "__main__":
    df, summary, ratings, genres, types = main()

# Additional utility functions for paper
def export_figures_for_paper():
    """Export high-quality figures for academic paper"""
    # Set publication style
    plt.rcParams.update({
        'font.size': 12,
        'font.family': 'serif',
        'figure.dpi': 300,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight'
    })
    
    print("📸 Exporting publication-quality figures...")
    
    # Re-run plots with publication settings
    df = load_and_merge_data()
    
    # Cluster sizes for paper
    plt.figure(figsize=(10, 6))
    plot_cluster_sizes(df)
    plt.savefig('cluster_sizes_publication.png', dpi=300, bbox_inches='tight')
    
    # Genre heatmap for paper
    plt.figure(figsize=(12, 8))
    plot_genre_heatmap(df)
    plt.savefig('genre_heatmap_publication.png', dpi=300, bbox_inches='tight')
    
    print("✅ Figures exported to current directory")

# Uncomment to export publication figures
# export_figures_for_paper()

🎌 ANIME CLUSTER CHARACTERIZATION ANALYSIS
Loading data...


FileNotFoundError: [Errno 2] No such file or directory: '../data/final_clusters/spectral_13_reps.csv'